这份代码是**推荐系统的「排序阶段」完整实现**，承接前面的**召回环节**，核心是给用户的候选文章精准打分、排序，最终输出每人Top5推荐文章。

### 一、核心定位
- 召回：**粗筛**，从海量文章里快速筛出几百个候选，缩小范围。
- 排序：**精排**，对候选集精准计算点击概率，按概率从高到低排序，选出最贴合用户的5篇。

### 二、主要做了这4件事
1. **读取并预处理排序特征**
    加载用户、文章、用户-文章交互的特征（相似度、时间差、点击历史、类别等），清洗数据、规整格式。
2. **训练3种经典排序模型打分**
    - **LGB排序模型**：专门做排序任务，优化推荐排序指标（NDCG）。
    - **LGB分类模型**：把“是否点击”转为二分类，输出点击概率。
    - **DIN深度模型**：阿里的深度兴趣网络，捕捉用户历史行为与候选文章的相关性，更精准刻画用户兴趣。
3. **5折交叉验证**
    按用户划分折数，防止数据泄露，输出模型预测分数/排名，用于后续融合。
4. **模型融合提升效果**
    - 加权融合：直接累加3个模型的预测分数，综合排序。
    - Stacking：用逻辑回归拟合3个模型的输出，二次预测更稳。

### 三、最终产出
生成**每个用户Top5推荐文章**的CSV提交文件，是推荐系统的最终输出结果。

简单说：你接下来要做的，就是**用特征训练排序模型→融合模型→输出最终推荐列表**。

In [2]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import gc, os
import time
from datetime import datetime
import lightgbm as lgb
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

In [3]:
data_path = './data_raw/'
save_path = './temp_results/'
offline = False

In [4]:
# 重新读取数据的时候，发现click_article_id是一个浮点数，所以将其转换成int类型
trn_user_item_feats_df = pd.read_csv(save_path + 'trn_user_item_feats_df.csv')
trn_user_item_feats_df['click_article_id'] = trn_user_item_feats_df['click_article_id'].astype(int)

if offline:
    val_user_item_feats_df = pd.read_csv(save_path + 'val_user_item_feats_df.csv')
    val_user_item_feats_df['click_article_id'] = val_user_item_feats_df['click_article_id'].astype(int)
else:
    val_user_item_feats_df = None
    
tst_user_item_feats_df = pd.read_csv(save_path + 'tst_user_item_feats_df.csv')
tst_user_item_feats_df['click_article_id'] = tst_user_item_feats_df['click_article_id'].astype(int)

# 做特征的时候为了方便，给测试集也打上了一个无效的标签，这里直接删掉就行
del tst_user_item_feats_df['label']

### 本部分是推荐系统**排序阶段收尾**的两个核心工具函数，分工明确：
1. submit()：将模型预测的排序结果，按用户提取Top5推荐文章，转换成标准提交格式并保存CSV文件；
2. norm_sim()：对预测分数做**最小-最大归一化**，把分数缩放到0~1区间，消除不同模型分数量级差异，为模型融合做准备。

### norm_sim() 函数的使用场景与核心作用
一句话总结：**这个函数是为「模型融合」服务的**，解决不同模型输出分数「量级不统一、无法直接相加/融合」的问题。     

推荐系统里我们训练了 3 个不同的排序模型，它们输出的 pred_score（预测分数）范围完全不一样：         
LGB 分类模型：输出是 0~1 之间的概率值（比如 0.7 表示 70% 点击概率）；         
LGB 排序模型：输出是「相对排序分」，范围可能是 -5~5 或者 0~100（没有固定上下限）；    
DIN 深度模型：输出同样是 0~1 概率，但分布可能和 LGB 分类模型不一样。    
如果直接把这 3 个分数加起来做融合，会出现一个问题：               
比如 LGB 排序模型的分数是 0~100，另外两个是 0~1，直接相加的话，LGB 排序模型的分数会「主导」结果，另外两个模型的影响几乎被淹没，融合就没意义了。          
norm_sim () 的作用：把所有模型的分数统一缩放到 0~1 之间，让每个模型的「话语权」平等，融合才有效。      

场景 1：加权融合前（直接对预测分数归一化）       
看代码最后的「模型融合 - 加权融合」部分：          
场景 2：五折交叉验证后（对 Stacking 特征归一化）      
看 LGB 排序模型的五折交叉验证部分：

In [5]:
# 生成标准提交结果函数
def submit(recall_df, topk=5, model_name=None):
    # 按用户ID分组，对预测分数升序排序（为后续倒序排名做准备）
    recall_df = recall_df.sort_values(by=['user_id', 'pred_score'])
    # 按用户分组，对预测分数降序排名，method='first'保证相同分数不并列
    recall_df['rank'] = recall_df.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')
    
    # 校验：确保每个用户至少有topk篇候选文章，不满足直接报错
    tmp = recall_df.groupby('user_id').apply(lambda x: x['rank'].max())
    assert tmp.min() >= topk
    
    # 删除预测分数列，仅保留ID和排名
    del recall_df['pred_score']
    # 筛选每个用户Topk的文章，行转列（宽表格式）适配提交规范
    submit = recall_df[recall_df['rank'] <= topk].set_index(['user_id', 'rank']).unstack(-1).reset_index()
    
    # 格式化列名，清理多层索引
    submit.columns = [int(col) if isinstance(col, int) else col for col in submit.columns.droplevel(0)]
    # 按照标准提交格式重命名列
    submit = submit.rename(columns={'': 'user_id', 1: 'article_1', 2: 'article_2', 
                                                  3: 'article_3', 4: 'article_4', 5: 'article_5'})
    
    # 拼接保存路径+文件名，导出CSV文件
    save_name = save_path + model_name + '_' + datetime.today().strftime('%m-%d') + '.csv'
    submit.to_csv(save_name, index=False, header=True)

# 预测分数归一化函数（用于模型融合）
def norm_sim(sim_df, weight=0.0):
    # 计算分数的最小值和最大值
    min_sim = sim_df.min()
    max_sim = sim_df.max()
    # 特殊情况：所有分数相同，直接赋值为1.0
    if max_sim == min_sim:
        sim_df = sim_df.apply(lambda sim: 1.0)
    # 常规情况：最小-最大归一化，将分数缩放到0~1
    else:
        sim_df = sim_df.apply(lambda sim: 1.0 * (sim - min_sim) / (max_sim - min_sim))

    # 给归一化后的分数加偏移权重（可选）
    sim_df = sim_df.apply(lambda sim: sim + weight)
    return sim_df

# LGB

## 一、LGB（LightGBM）是什么？
一句话：**一个非常好用的「机器学习模型」，专门用来做预测/排序，速度快、效果好**。
- 类比：就像一个「超级做题家」，你给它一堆题目（特征）和答案（标签），它能快速学会规律，然后做新题（预测）。
- 优势：比传统模型快几十倍，能处理大量数据，推荐系统里的「标配模型」。

---

## 二、LGB排序模型 vs LGB分类模型，分别干啥？
推荐系统里，我们的目标是「给用户推荐最可能点击的文章」，有两种实现思路：

### 1. LGB分类模型（Classification）
- **思路**：把问题变成「判断题」——这篇文章用户「会点击」还是「不会点击」？
- **输出**：0~1之间的概率值（比如 0.8 表示 80% 概率点击）。
- **类比**：老师给试卷打分，0分（不点击）到100分（点击），分数越高越推荐。

### 2. LGB排序模型（Ranking）
- **思路**：把问题变成「排序题」——给用户的几篇候选文章，按「点击可能性」从高到低排个序。
- **输出**：相对分数（没有固定范围，只看相对大小）。
- **类比**：运动会跑步比赛，不看具体跑了多少秒，只看谁是第一、谁是第二，按名次推荐。
- **核心区别**：排序模型更关注「相对顺序」，这正是推荐系统最需要的（我们只关心哪篇文章排第一，不关心具体分数是多少）。

---

## 三、单模型训练 vs 五折交叉验证，分别干啥？
### 1. 单模型训练
- **思路**：用全部训练数据训练一个模型，直接预测测试集。
- **作用**：快速跑通流程，看模型效果，生成一个基础的推荐结果。
- **类比**：平时做一套练习题，直接上考场。

### 2. 五折交叉验证
- **思路**：把训练数据分成5份，每次用4份训练、1份验证，重复5次，最后把5次的结果平均。
- **作用**：
  1. 更准确地评估模型效果（避免一次运气好/不好）；
  2. 生成「模型融合」需要的新特征（Stacking用）。
- **类比**：平时做5套练习题，每次模拟考试，最后取平均成绩，更稳。

---

## 四、你接下来会看到的代码流程
1. **单模型训练**：先训练一个LGB排序模型，生成推荐结果；
2. **五折交叉验证**：用更严谨的方式再训练一次LGB排序模型，生成融合用的特征；
3. **LGB分类模型**：换一种思路（分类），再训练一个模型；
4. **DIN深度模型**：用深度学习再训练一个模型；
5. **模型融合**：把3个模型的结果结合起来，得到最终的推荐。

## **LGB排序模型的单模型训练**

In [ ]:
# 核心流程：数据备份→定义特征→按用户分组→定义模型→训练→预测→保存结果→生成提交文件

# 1. 数据备份：防止修改原始数据，重新读取麻烦
trn_user_item_feats_df_rank_model = trn_user_item_feats_df.copy()
if offline:
    val_user_item_feats_df_rank_model = val_user_item_feats_df.copy()
tst_user_item_feats_df_rank_model = tst_user_item_feats_df.copy()

# 2. 定义特征列：告诉模型用哪些数据来预测
lgb_cols = ['sim0', 'time_diff0', 'word_diff0','sim_max', 'sim_min', 'sim_sum', 
            'sim_mean', 'score','click_size', 'time_diff_mean', 'active_level',
            'click_environment','click_deviceGroup', 'click_os', 'click_country', 
            'click_region','click_referrer_type', 'user_time_hob1', 'user_time_hob2',
            'words_hbo', 'category_id', 'created_at_ts','words_count']

# 3. 排序模型分组：按用户分组，这是LGB排序模型的特殊要求
# 按用户 ID 排序，并统计每个用户有多少篇候选文章
# LGB 排序模型是「按用户」学习的，它需要知道哪些文章属于同一个用户，才能给这个用户的文章排顺序
trn_user_item_feats_df_rank_model.sort_values(by=['user_id'], inplace=True)
g_train = trn_user_item_feats_df_rank_model.groupby(['user_id'], as_index=False).count()["label"].values
if offline:
    val_user_item_feats_df_rank_model.sort_values(by=['user_id'], inplace=True)
    g_val = val_user_item_feats_df_rank_model.groupby(['user_id'], as_index=False).count()["label"].values

# 4. 定义LGB排序模型：设置模型参数
lgb_ranker = lgb.LGBMRanker(boosting_type='gbdt', num_leaves=31, reg_alpha=0.0, reg_lambda=1,
                            max_depth=-1, n_estimators=100, subsample=0.7, colsample_bytree=0.7, subsample_freq=1,
                            learning_rate=0.01, min_child_weight=50, random_state=2018, n_jobs=16)  
        # boosting_type='gbdt'：用梯度提升树（最常用的类型）；
        # num_leaves=31：每棵树的叶子节点数（控制模型复杂度）；
        # learning_rate=0.01：学习率（越小越稳，但训练越慢）；
        # n_estimators=100：树的数量（越多越准，但可能过拟合）；
        # reg_alpha=0.0, reg_lambda=1：正则化参数（防止过拟合）；
        # random_state=2018：随机种子（保证每次运行结果一样）；
        # n_jobs=16：用 16 个 CPU 核心并行计算（速度快）。

# 5. 训练模型：分离线调试和正式训练两种模式
if offline:
    # 离线模式 用ndcg指标评估 验证集效果50轮没提升则提前停止
    lgb_ranker.fit(trn_user_item_feats_df_rank_model[lgb_cols], trn_user_item_feats_df_rank_model['label'], group=g_train,
                eval_set=[(val_user_item_feats_df_rank_model[lgb_cols], val_user_item_feats_df_rank_model['label'])], 
                eval_group=[g_val], eval_at=[1, 2, 3, 4, 5], eval_metric=['ndcg'], early_stopping_rounds=50)
else:
    # 正式模式 无验证集，全部训练数据训练
    lgb_ranker.fit(trn_user_item_feats_df[lgb_cols], trn_user_item_feats_df['label'], group=g_train)

# 6. 模型预测：对测试集打分
# num_iteration=lgb_ranker.best_iteration_：用效果最好的那一轮树来预测（离线模式下有用，正式模式下用全部树）
tst_user_item_feats_df['pred_score'] = lgb_ranker.predict(tst_user_item_feats_df[lgb_cols], num_iteration=lgb_ranker.best_iteration_)

# 7. 保存预测结果：留着后面模型融合用
tst_user_item_feats_df[['user_id', 'click_article_id', 'pred_score']].to_csv(save_path + 'lgb_ranker_score.csv', index=False)

# 8. 生成提交结果：调用前面的 submit 函数，生成Top5推荐
rank_results = tst_user_item_feats_df[['user_id', 'click_article_id', 'pred_score']]
rank_results['click_article_id'] = rank_results['click_article_id'].astype(int)
submit(rank_results, topk=5, model_name='lgb_ranker')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001495 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4129
[LightGBM] [Info] Number of data points in the train set: 291135, number of used features: 23
